In [2]:
import mlflow
import pandas as pd
import mlflow.sklearn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
import re
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import numpy as np
import nltk

In [3]:
df = pd.read_csv('IMDB.csv')
df = df.sample(500)
df.to_csv('data.csv', index=False)
df.head()

,review,sentiment
12,"For me, North and South (Books I&II) is the ul...",positive
52,This is an EXCELLENT example of early Bette Da...,positive
627,This movie will always be a Broadway and Movie...,positive
51,"Admittedly, Parsifal is not an opera that can ...",negative
258,In the opening scenes of this movie a man shot...,positive


In [4]:
# data preprocessing

# Define text preprocessing functions
def lemmatization(text):
    """Lemmatize the text."""
    lemmatizer = WordNetLemmatizer()
    text = text.split()
    text = [lemmatizer.lemmatize(word) for word in text]
    return " ".join(text)

def remove_stop_words(text):
    """Remove stop words from the text."""
    stop_words = set(stopwords.words("english"))
    text = [word for word in str(text).split() if word not in stop_words]
    return " ".join(text)

def removing_numbers(text):
    """Remove numbers from the text."""
    text = ''.join([char for char in text if not char.isdigit()])
    return text

def lower_case(text):
    """Convert text to lower case."""
    text = text.split()
    text = [word.lower() for word in text]
    return " ".join(text)

def removing_punctuations(text):
    """Remove punctuations from the text."""
    text = re.sub('[%s]' % re.escape(string.punctuation), ' ', text)
    text = text.replace('؛', "")
    text = re.sub('\s+', ' ', text).strip()
    return text

def removing_urls(text):
    """Remove URLs from the text."""
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)

def normalize_text(df):
    """Normalize the text data."""
    try:
        df['review'] = df['review'].apply(lower_case)
        df['review'] = df['review'].apply(remove_stop_words)
        df['review'] = df['review'].apply(removing_numbers)
        df['review'] = df['review'].apply(removing_punctuations)
        df['review'] = df['review'].apply(removing_urls)
        df['review'] = df['review'].apply(lemmatization)
        return df
    except Exception as e:
        print(f'Error during text normalization: {e}')
        raise

In [5]:
try:
    nltk.data.find('corpora/wordnet.zip')
except LookupError:
    nltk.download('wordnet')

df = normalize_text(df)
df.head()

,review,sentiment
12,me north south book i ii ultimate tv series s ...,positive
52,excellent example early bette davis talent pro...,positive
627,movie always broadway movie classic long still...,positive
51,admittedly parsifal opera appeal everyone alth...,negative
258,opening scene movie man shot arrow hotel room ...,positive


In [6]:
df['sentiment'].value_counts()

sentiment
negative    253
positive    247
Name: count, dtype: int64

In [7]:
x = df['sentiment'].isin(['positive','negative'])
df = df[x]

In [8]:
df['sentiment'] = df['sentiment'].map({'positive':1, 'negative':0})
df.head()

,review,sentiment
12,me north south book i ii ultimate tv series s ...,1
52,excellent example early bette davis talent pro...,1
627,movie always broadway movie classic long still...,1
51,admittedly parsifal opera appeal everyone alth...,0
258,opening scene movie man shot arrow hotel room ...,1


In [9]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [16]:
vectorizer = CountVectorizer(max_features=50)
X = vectorizer.fit_transform(df['review'])
y = df['sentiment']

In [17]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [14]:
import dagshub

mlflow.set_tracking_uri('https://dagshub.com/rehansarfraz8903/NLP-Sentiment-Analysis.mlflow')
dagshub.init(repo_owner='rehansarfraz8903', repo_name='NLP-Sentiment-Analysis', mlflow=True)

# mlflow.set_experiment("Logistic Regression Baseline")
mlflow.set_experiment("Logistic Regression Baseline")


Initialized MLflow to track repo "rehansarfraz8903/NLP-Sentiment-Analysis"

Repository rehansarfraz8903/NLP-Sentiment-Analysis initialized!

2026/07/26 16:43:40 INFO mlflow.tracking.fluent: Experiment with name 'Logistic Regression Baseline' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/3aa39314432447bc935a93ccf1cf9c4c', creation_time=1785066234809, effective_trace_archival_retention=None, experiment_id='0', last_update_time=1785066234809, lifecycle_stage='active', name='Logistic Regression Baseline', tags={}, trace_location=None, workspace='default'>

In [18]:
import mlflow
import logging
import os
import time
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Configure logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

logging.info("Starting MLflow run...")

with mlflow.start_run():
    start_time = time.time()
    
    try:
        logging.info("Logging preprocessing parameters...")
        mlflow.log_param("vectorizer", "Bag of Words")
        mlflow.log_param("num_features", 50)
        mlflow.log_param("test_size", 0.2)

        logging.info("Initializing Logistic Regression model...")
        model = LogisticRegression(max_iter=1000)  # Increase max_iter to prevent non-convergence issues

        logging.info("Fitting the model...")
        model.fit(X_train, y_train)
        logging.info("Model training complete.")

        logging.info("Logging model parameters...")
        mlflow.log_param("model", "Logistic Regression")

        logging.info("Making predictions...")
        y_pred = model.predict(X_test)

        logging.info("Calculating evaluation metrics...")
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        logging.info("Logging evaluation metrics...")
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1_score", f1)

        logging.info("Saving and logging the model...")
        mlflow.sklearn.log_model(model, "model")

        # Log execution time
        end_time = time.time()
        logging.info(f"Model training and logging completed in {end_time - start_time:.2f} seconds.")

        # Save and log the notebook
        # notebook_path = "exp1_baseline_model.ipynb"
        # logging.info("Executing Jupyter Notebook. This may take a while...")
        # os.system(f"jupyter nbconvert --to notebook --execute --inplace {notebook_path}")
        # mlflow.log_artifact(notebook_path)

        # logging.info("Notebook execution and logging complete.")

        # Print the results for verification
        logging.info(f"Accuracy: {accuracy}")
        logging.info(f"Precision: {precision}")
        logging.info(f"Recall: {recall}")
        logging.info(f"F1 Score: {f1}")

    except Exception as e:
        logging.error(f"An error occurred: {e}", exc_info=True)


2026-07-26 16:52:57,903 - INFO - Starting MLflow run...


2026-07-26 16:52:59,426 - INFO - Logging preprocessing parameters...
2026-07-26 16:53:00,796 - INFO - Initializing Logistic Regression model...
2026-07-26 16:53:00,799 - INFO - Fitting the model...
2026-07-26 16:53:00,988 - INFO - Model training complete.
2026-07-26 16:53:00,991 - INFO - Logging model parameters...
2026-07-26 16:53:01,441 - INFO - Making predictions...
2026-07-26 16:53:01,448 - INFO - Calculating evaluation metrics...
2026-07-26 16:53:01,580 - INFO - Logging evaluation metrics...
2026-07-26 16:53:03,386 - INFO - Saving and logging the model...
2026/07/26 16:53:03 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026-07-26 16:53:34,483 - INFO - Model training and logging completed in 35.06 seconds.
2026-07-26 16:53:34,488 - INFO - Accuracy: 0.63
2026-07-26 16:53:34,492 - INFO - Precision: 0.6829268292682927
2026-07-26 16:53:34,503 - INFO - Recall: 0.5384615384615384
2026-07-26 16:53:34,512 - INFO - F1 Score: 0.6021505376344086


🏃 View run dapper-dolphin-85 at: https://dagshub.com/rehansarfraz8903/NLP-Sentiment-Analysis.mlflow/#/experiments/0/runs/0b0f3f8027554ed4814202f75c949c9d
🧪 View experiment at: https://dagshub.com/rehansarfraz8903/NLP-Sentiment-Analysis.mlflow/#/experiments/0
